# SARIMA：带季节性的 ARIMA
> 难度标签：中级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：10_时间序列 | 源文件：SARIMA.py | 核心功能：季节性差分、季节性参数、周期性建模

## 概述

SARIMA 在 ARIMA 基础上加入季节性分量（P, D, Q, s），能建模周期性规律（如月度销售额、日温度变化）。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import seasonal_decompose
# --- 导入库 ---
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.metrics import mean_squared_error, mean_absolute_error

## 数学原理

### 1. SARIMA(p,d,q)(P,D,Q,s) 模型

SARIMA 在 ARIMA 基础上增加季节性分量，记为 $\text{SARIMA}(p,d,q)(P,D,Q,s)$：

$$(1 - \sum_{i=1}^p \phi_i B^i)(1 - \sum_{I=1}^P \Phi_I B^{Is})(1-B)^d(1-B^s)^D y_t$$

$$= (1 + \sum_{j=1}^q \theta_j B^j)(1 + \sum_{J=1}^Q \Theta_J B^{Js})\epsilon_t$$

其中 $B$ 是后移算子（$B y_t = y_{t-1}$），$s$ 是季节周期。

### 2. 季节性分解

加法分解模型：

$$y_t = T_t + S_t + R_t$$

- $T_t$：趋势成分（长期变化方向）
- $S_t$：季节成分（固定周期重复模式），$\sum_{j=0}^{s-1} S_{t+j} = 0$
- $R_t$：残差成分（随机波动）

乘法分解模型：$y_t = T_t \times S_t \times R_t$

### 3. 季节性差分

**一阶季节差分**（周期 $s$）：

$$\nabla_s y_t = y_t - y_{t-s}$$

消除季节性。代码中 SARIMA 的 $D$ 参数控制季节性差分阶数。

**组合差分**：同时做普通差分和季节差分

$$\nabla \nabla_s y_t = (y_t - y_{t-1}) - (y_{t-s} - y_{t-s-1})$$

### 4. ADF 检验与平稳性

对差分后的序列进行 ADF 检验，确认平稳性：

$$H_0: \text{序列非平稳}, \quad H_1: \text{序列平稳}$$

$p < 0.05$ 拒绝 $H_0$，序列可建模。

### 5. 参数选择

| 参数 | 含义 | 典型值 |
|------|------|--------|
| $p$ | 非季节 AR 阶数 | 0-3 |
| $d$ | 普通差分阶数 | 0-2 |
| $q$ | 非季节 MA 阶数 | 0-3 |
| $P$ | 季节 AR 阶数 | 0-2 |
| $D$ | 季节差分阶数 | 0-1 |
| $Q$ | 季节 MA 阶数 | 0-2 |
| $s$ | 季节周期 | 12（月）、7（周）、365（日） |

### 6. 预测评估指标

**MSE**（均方误差）：$\text{MSE} = \frac{1}{n}\sum_{i=1}^n (y_i - \hat{y}_i)^2$

**MAE**（平均绝对误差）：$\text{MAE} = \frac{1}{n}\sum_{i=1}^n |y_i - \hat{y}_i|$

**MAPE**（平均绝对百分比误差）：$\text{MAPE} = \frac{100\%}{n}\sum_{i=1}^n \left|\frac{y_i - \hat{y}_i}{y_i}\right|$

### 7. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `SARIMAX(序列, order=(p,d,q), seasonal_order=(P,D,Q,s))` | SARIMA 模型 |
| `seasonal_decompose(序列, model="additive", period=365)` | 加法季节分解 $y = T + S + R$ |
| `model.fit(disp=False)` | 最大似然估计参数 |
| `forecast(steps=len(test))` | 多步预测 |
| `mean_squared_error(test, pred)` | MSE 评估 |
| `acorr_ljungbox(resid)` | 残差白噪声检验 |

### 1. 生成具有季节性的合成时间序列

运行 1. 生成具有季节性的合成时间序列 的代码，观察算法在该环节的行为。

In [ ]:
np.random.seed(42)
n = 365 * 2  # 两年日数据

t = np.arange(n)
趋势 = 0.02 * t
年度季节 = 3 * np.sin(2 * np.pi * t / 365)  # 年周期
周季节 = 0.8 * np.sin(2 * np.pi * t / 7)    # 周周期
噪声 = np.random.normal(0, 0.6, n)
# --- 赋值/配置 ---
数据 = 趋势 + 年度季节 + 周季节 + 噪声

序列 = pd.Series(数据, index=pd.date_range("2022-01-01", periods=n, freq="D"))
print("=" * 55)
print("SARIMA 季节性时间序列建模")
print("=" * 55)
print(f"序列长度: {n} 天（{序列.index[0].date()} ~ {序列.index[-1].date()}）")
# --- 输出结果 ---
print(f"均值: {序列.mean():.4f}, 标准差: {序列.std():.4f}")

### 2. 季节性分解

运行 2. 季节性分解 的代码，观察算法在该环节的行为。

In [ ]:
print("\n--- 季节性分解（周期=365） ---")
分解 = seasonal_decompose(序列, model="additive", period=365)
趋势成分 = 分解.trend.dropna()
季节成分 = 分解.seasonal.dropna()
残差成分 = 分解.resid.dropna()

print(f"  趋势成分范围: [{趋势成分.min():.3f}, {趋势成分.max():.3f}]")
print(f"  季节成分范围: [{季节成分.min():.3f}, {季节成分.max():.3f}]")
print(f"  残差标准差:   {残差成分.std():.4f}")

### 3. 平稳性检验

检验时间序列的平稳性，决定差分阶数。

In [ ]:
def adf检验(序列, 名称="序列"):
    result = adfuller(序列.dropna(), autolag="AIC")
    平稳 = result[1] < 0.05
    print(f"  ADF 检验 ({名称}): 统计量={result[0]:.4f}, p={result[1]:.4f} → {'平稳' if 平稳 else '非平稳'}")
    return 平稳

print("\n--- 平稳性检验 ---")
adf检验(序列, "原始序列")
差分1 = 序列.diff().dropna()
adf检验(差分1, "一阶差分")
差分季节 = 序列.diff(365).dropna()
# --- 继续 ---
adf检验(差分季节, "季节差分(s=365)")

### 4. SARIMA 模型拟合

运行 4. SARIMA 模型拟合 的代码，观察算法在该环节的行为。

In [ ]:
print("\n--- 模型拟合 ---")
train_size = int(n * 0.8)
train, test = 序列[:train_size], 序列[train_size:]
print(f"训练集: {len(train)} 天, 测试集: {len(test)} 天")

# 参数选择策略：使用较小的参数组合避免过拟合
# 非季节性 (p,d,q) = (1,1,1), 季节性 (P,D,Q,s) = (1,1,1,7)
order = (1, 1, 1)
seasonal_order = (1, 1, 1, 7)

print(f"参数: SARIMA{order}x{seasonal_order}")
模型 = SARIMAX(train, order=order, seasonal_order=seasonal_order,
               enforce_stationarity=False, enforce_invertibility=False)
拟合结果 = 模型.fit(disp=False, maxiter=200)

print(f"  AIC:  {拟合结果.aic:.2f}")
print(f"  BIC:  {拟合结果.bic:.2f}")

### 5. 残差诊断

检查模型残差是否满足独立同分布假设。

In [ ]:
残差 = 拟合结果.resid
print(f"\n--- 残差诊断 ---")
print(f"  均值:   {残差.mean():.6f}")
print(f"  标准差: {残差.std():.4f}")

lb = acorr_ljungbox(残差.dropna(), lags=[10, 20], return_df=True)
for lag, row in lb.iterrows():
    p = row["lb_pvalue"]
    print(f"  Ljung-Box (lag={lag}): p={p:.4f} → {'无自相关' if p > 0.05 else '存在自相关'}")

### 6. 预测与评估

使用训练好的模型进行预测，观察输出结果。

In [ ]:
forecast = 拟合结果.forecast(steps=len(test))
rmse = np.sqrt(mean_squared_error(test, forecast))
mae = mean_absolute_error(test, forecast)
mape = np.mean(np.abs((test.values - forecast.values) / test.values)) * 100

print(f"\n--- 预测评估 ---")
print(f"  RMSE: {rmse:.4f}")
print(f"  MAE:  {mae:.4f}")
print(f"  MAPE: {mape:.2f}%")

print(f"\n--- 预测 vs 真实（前 10 个） ---")
print(f"{'日期':>12}  {'真实':>8}  {'预测':>8}  {'误差':>8}")
for i in range(min(10, len(test))):
    print(f"  {str(test.index[i].date()):>10}  {test.values[i]:>8.3f}  {forecast.values[i]:>8.3f}  {test.values[i] - forecast.values[i]:>8.3f}")

### 7. 全序列拟合 + 未来预测

使用训练好的模型进行预测，观察输出结果。

In [ ]:
print("\n--- 全序列拟合 & 未来 30 天预测 ---")
全模型 = SARIMAX(序列, order=order, seasonal_order=seasonal_order,
                 enforce_stationarity=False, enforce_invertibility=False)
全拟合 = 全模型.fit(disp=False, maxiter=200)
未来预测 = 全拟合.forecast(steps=30)

print(f"{'日期':>12}  {'预测值':>8}")
for i in range(30):
    print(f"  {str(未来预测.index[i].date()):>10}  {未来预测.values[i]:>8.3f}")

## 关键代码解释

```python
from statsmodels.tsa.statespace.sarimax import SARIMAX
model = SARIMAX(ts, order=(1,1,1), seasonal_order=(1,1,1,12))
```

`s=12` 表示季节周期为 12（月度数据的年周期）。

## 注意事项

1. 季节周期 s 需要根据数据频率设定
2. 参数多，调参困难
3. 长期预测效果差（统计模型的通病）

## 总结与延伸

以上代码展示了 **SARIMA** 的完整流程。

**解读要点**：
- 观察预测曲线与实际值的 **趋势一致性**
- 关注残差是否呈现随机分布（无明显模式）
- 对比不同模型的 **MAPE / RMSE** 指标

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **ETS 模型**：指数平滑状态空间模型
- **TBATS**：多季节性时间序列
- **机器学习方法**：XGBoost、LSTM 对时序建模
